In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
DATA_PATH = "/content/drive/MyDrive/BCI_IV_2a"

In [2]:
# ============================================================
# CELL 1 (DEFINITIVE): Install for Colab
# Strategy: Accept Colab's NumPy/scipy/torch. Only install
# packages that are NOT pre-installed. Never force-reinstall
# NumPy or scipy — it breaks the entire runtime.
# ============================================================

# Step 1: Install MNE — has wheels for NumPy 2.x natively
!pip install "mne==1.9.0" -q

# Step 2: Install scikit-learn — pre-built for NumPy 2.x
!pip install "scikit-learn==1.6.1" -q

# Step 3: Install pandas — pin back to what google-colab requires
!pip install "pandas==2.2.2" -q

# Step 4: Install MOABB — MUST install after pinning pandas
# Use 1.2.0 which has pre-built wheels and supports NumPy 2.x
!pip install "moabb==1.2.0" -q

# Step 5: Utilities
!pip install matplotlib seaborn tqdm -q

# ── Do NOT touch: numpy, scipy, torch ─────────────────────
# Colab already has working versions. Reinstalling them
# breaks CUDA linkage and causes the _center import error.

print("\n✅ Installation complete.")
print("⚠️  NOW: Runtime > Restart session > then run Cell 1b below")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 79.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
moabb 1.4.3 requires mne>=1.10.0, but you have mne 1.9.0 which is incompatible.
moabb 1.4.3 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
braindecode 1.3.2 requires mne>=1.11.0, but you have mne 1.9.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 94.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
moabb 1.4.3 requires mne>=1.10.0, but you have mne 1.9.0 which is incompatible.
braindecode 1.3.2 requires mne>=1.11.0, but you have mne 1.9.0 which is incompatible.
tsfresh 0.21.1 requires scipy>=1.14.0; python_version >= "3.10", bu

In [1]:
# ============================================================
# CELL 1b: Verify Environment (run after restarting runtime)
# ============================================================
import sys, importlib

packages = ["numpy", "scipy", "pandas", "torch", "mne", "moabb", "sklearn",
            "matplotlib", "seaborn", "tqdm"]

print(f"Python : {sys.version.split()[0]}\n")
print(f"{'Package':<14} {'Version':<15}")
print("-" * 30)

all_ok = True
for pkg in packages:
    mod = "sklearn" if pkg == "sklearn" else pkg
    try:
        m = importlib.import_module(mod)
        print(f"{pkg:<14} {m.__version__:<15} ✅")
    except Exception as e:
        print(f"{pkg:<14} FAILED ❌  {e}")
        all_ok = False

import torch
print(f"\n{'CUDA':<14} {str(torch.cuda.is_available()):<15} "
      f"{'✅ ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else '⚠️  No GPU'}")

print("\n" + ("✅ All good — proceed to Cell 2." if all_ok
              else "❌ Fix errors above before continuing."))

Python : 3.12.12

Package        Version        
------------------------------
numpy          1.26.4          ✅
scipy          1.13.1          ✅
pandas         2.2.2           ✅
torch          2.10.0+cu128    ✅
mne            1.9.0           ✅
moabb          1.2.0           ✅
sklearn        1.5.2           ✅
matplotlib     3.10.8          ✅
seaborn        0.12.2          ✅
tqdm           4.67.3          ✅

CUDA           True            ✅ Tesla T4

✅ All good — proceed to Cell 2.


In [2]:
# ============================================================
# CELL 2: GPU Verification
# ============================================================
import torch

print(f"PyTorch    : {torch.__version__}")
print(f"CUDA avail : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU        : {torch.cuda.get_device_name(0)}")
    print(f"VRAM       : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU — Go to Runtime > Change runtime type > T4 GPU")

PyTorch    : 2.10.0+cu128
CUDA avail : True
GPU        : Tesla T4
VRAM       : 15.6 GB


In [3]:
# ============================================================
# CELL 3 (DEFINITIVE): All Imports
# Verified stack: Python 3.12 | NumPy 2.4.x | SciPy 1.17
#                 PyTorch 2.10 | MNE 1.9 | MOABB 1.2 | sklearn 1.6
# ============================================================
import os, sys, random, logging, warnings
from pathlib import Path
from typing import Tuple, Dict, List, Optional

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# ── Numerical ─────────────────────────────────────────────
import numpy as np
import scipy.signal as signal
import pandas as pd

# ── Plotting ──────────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# ── Progress ──────────────────────────────────────────────
from tqdm.notebook import tqdm

# ── MNE ───────────────────────────────────────────────────
import mne
mne.set_log_level("ERROR")

# ── MOABB ─────────────────────────────────────────────────
# MOABB 1.2.0 uses underscore naming: BNCI2014_001
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery

# ── PyTorch ───────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

# ── Scikit-Learn ──────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    cohen_kappa_score, accuracy_score,
)
from sklearn.model_selection import StratifiedKFold

# ── Logger ────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s — %(levelname)s — %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

# ── Global Seed ───────────────────────────────────────────
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ["PYTHONHASHSEED"]       = str(seed)

seed_everything(42)

# ── Summary ───────────────────────────────────────────────
print("=" * 46)
print("  ENVIRONMENT VERIFIED")
print("=" * 46)
print(f"  Python     : {sys.version.split()[0]}")
print(f"  NumPy      : {np.__version__}")
print(f"  Pandas     : {pd.__version__}")
print(f"  PyTorch    : {torch.__version__}")
print(f"  MNE        : {mne.__version__}")
import moabb, sklearn
print(f"  MOABB      : {moabb.__version__}")
print(f"  Sklearn    : {sklearn.__version__}")
print(f"  Device     : {'CUDA — ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU ⚠️'}")
print("=" * 46)
print("✅ All imports successful. Proceed to Cell 4.")

  ENVIRONMENT VERIFIED
  Python     : 3.12.12
  NumPy      : 1.26.4
  Pandas     : 2.2.2
  PyTorch    : 2.10.0+cu128
  MNE        : 1.9.0
  MOABB      : 1.2.0
  Sklearn    : 1.5.2
  Device     : CUDA — Tesla T4
✅ All imports successful. Proceed to Cell 4.


In [4]:
# ============================================================
# CELL 4: Global Configuration
# ============================================================
class Config:
    # ── Data ──────────────────────────────────────────────
    SUBJECTS      = list(range(1, 10))   # All 9 subjects
    N_CLASSES     = 4
    CLASS_NAMES   = ["Left Hand", "Right Hand", "Feet", "Tongue"]
    SFREQ         = 250                  # Hz
    T_MIN         = 0.5                  # seconds post-cue (skip motor prep artifact)
    T_MAX         = 4.0                  # seconds post-cue
    N_CHANNELS    = 22                   # EEG only (no EOG)
    N_SAMPLES     = int((T_MAX - T_MIN) * SFREQ)  # 875 samples

    # ── Preprocessing ─────────────────────────────────────
    BANDPASS_LOW  = 4.0
    BANDPASS_HIGH = 40.0
    NOTCH_FREQ    = 50.0

    # ── Enhanced EEGNet ───────────────────────────────────
    F1            = 16        # Increased temporal filters
    D             = 2         # Depth multiplier → F2 = F1*D = 32
    F2            = 32
    KERNEL_SIZE   = 64        # 256ms at 250Hz
    DROPOUT       = 0.4       # Slightly lower dropout for more complex model
    POOL1         = 4
    POOL2         = 8

    # ── Training ──────────────────────────────────────────
    BATCH_SIZE    = 32        # Smaller batch = better generalization on small EEG datasets
    EPOCHS        = 500
    LR            = 5e-4
    WEIGHT_DECAY  = 1e-3
    PATIENCE      = 60
    LR_PATIENCE   = 25
    LR_FACTOR     = 0.5
    MIN_LR        = 1e-6
    N_FOLDS       = 5
    SEED          = 42

    # ── Ensemble ──────────────────────────────────────────
    N_ENSEMBLE    = 5         # Train 5 models per subject

    # ── Paths ─────────────────────────────────────────────
    MODEL_DIR   = Path("/content/models")
    RESULTS_DIR = Path("/content/results")
    DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cfg = Config()
cfg.MODEL_DIR.mkdir(parents=True, exist_ok=True)
cfg.RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Config ready.")
print(f"   Device      : {cfg.DEVICE}")
print(f"   N_SAMPLES   : {cfg.N_SAMPLES}  ({cfg.T_MIN}s → {cfg.T_MAX}s)")
print(f"   F1/F2/D     : {cfg.F1}/{cfg.F2}/{cfg.D}")

✅ Config ready.
   Device      : cuda
   N_SAMPLES   : 875  (0.5s → 4.0s)
   F1/F2/D     : 16/32/2


In [5]:
# ============================================================
# CELL 5: Data Loading via MOABB
# ============================================================
from moabb.datasets import BNCI2014001
from moabb.paradigms import MotorImagery

class BCIDataLoader:
    """
    Loads BCI Competition IV 2a using MOABB.
    Session 1 = train | Session 2 = test (official protocol)
    """
    def __init__(self, config: Config):
        self.cfg  = config
        self.dataset = BNCI2014001()
        self.paradigm = MotorImagery(
            n_classes=4,
            fmin=config.BANDPASS_LOW,
            fmax=config.BANDPASS_HIGH,
            tmin=config.T_MIN,
            tmax=config.T_MAX - 1.0/config.SFREQ,
            resample=config.SFREQ,
        )

    def load_subject(self, subject_id: int):
        X, y, meta = self.paradigm.get_data(
            dataset=self.dataset,
            subjects=[subject_id],
            return_epochs=False
        )
        sessions  = meta["session"].values
        le        = LabelEncoder()
        y_enc     = le.fit_transform(y)

        # MOABB session naming
        s_names   = np.unique(sessions)
        train_idx = sessions == s_names[0]   # Session 1
        test_idx  = sessions == s_names[1]   # Session 2

        X_tr = X[train_idx].astype(np.float32)
        y_tr = y_enc[train_idx]
        X_te = X[test_idx].astype(np.float32)
        y_te = y_enc[test_idx]
        return X_tr, y_tr, X_te, y_te

    def load_all(self):
        data = {}
        for sid in tqdm(self.cfg.SUBJECTS, desc="Loading subjects"):
            try:
                X_tr, y_tr, X_te, y_te = self.load_subject(sid)
                data[sid] = dict(X_train=X_tr, y_train=y_tr, X_test=X_te, y_test=y_te)
                print(f"  ✅ Subject {sid:02d} | Train: {X_tr.shape} | Test: {X_te.shape}")
            except Exception as e:
                print(f"  ⚠️ Subject {sid:02d} failed: {e}")
        return data

print("Loading dataset — this may take 3-5 minutes on first run (downloads ~500MB)...")
loader   = BCIDataLoader(cfg)
all_data = loader.load_all()
print(f"\n✅ Loaded {len(all_data)} subjects.")

Loading dataset — this may take 3-5 minutes on first run (downloads ~500MB)...


Loading subjects:   0%|          | 0/9 [00:00<?, ?it/s]

  ✅ Subject 01 | Train: (288, 22, 875) | Test: (288, 22, 875)
  ✅ Subject 02 | Train: (288, 22, 875) | Test: (288, 22, 875)
  ✅ Subject 03 | Train: (288, 22, 875) | Test: (288, 22, 875)
  ✅ Subject 04 | Train: (288, 22, 875) | Test: (288, 22, 875)
  ✅ Subject 05 | Train: (288, 22, 875) | Test: (288, 22, 875)
  ✅ Subject 06 | Train: (288, 22, 875) | Test: (288, 22, 875)
  ✅ Subject 07 | Train: (288, 22, 875) | Test: (288, 22, 875)
  ✅ Subject 08 | Train: (288, 22, 875) | Test: (288, 22, 875)
  ✅ Subject 09 | Train: (288, 22, 875) | Test: (288, 22, 875)

✅ Loaded 9 subjects.


In [6]:
# ============================================================
# CELL 6: Advanced Preprocessing Pipeline
# ============================================================
class EEGPreprocessor:
    """
    High-performance preprocessing stack:
    1. Artifact rejection (±6σ trial-level)
    2. Exponential Moving Standardization (EMS) — online-compatible
    3. Channel-wise z-score (train statistics applied to test)
    4. Common Average Reference (CAR)
    """
    @staticmethod
    def common_average_reference(X: np.ndarray) -> np.ndarray:
        """X: (trials, channels, time) — subtract mean across channels"""
        return X - X.mean(axis=1, keepdims=True)

    @staticmethod
    def reject_bad_trials(X: np.ndarray, y: np.ndarray,
                          threshold: float = 100.0):
        """
        Remove trials where peak-to-peak amplitude exceeds threshold (µV).
        Assumes data is in µV range after MOABB loading.
        """
        ptp = X.max(axis=(1, 2)) - X.min(axis=(1, 2))
        good = ptp < threshold
        rejected = (~good).sum()
        if rejected > 0:
            print(f"    ⚠️  Rejected {rejected}/{len(X)} trials (peak-to-peak > {threshold}µV)")
        return X[good], y[good]

    @staticmethod
    def exponential_moving_standardize(X: np.ndarray,
                                        factor_new: float = 0.001,
                                        init_block: int = 100) -> np.ndarray:
        """
        EMS from Schirrmeister et al. (2017) — causal, online-compatible.
        Removes slow drifts while preserving oscillatory content.
        X: (trials, channels, time)
        """
        X_out = np.zeros_like(X)
        for trial in range(X.shape[0]):
            x = X[trial]  # (channels, time)
            # Init statistics from first `init_block` samples
            mean = x[:, :init_block].mean(axis=1, keepdims=True)
            var  = x[:, :init_block].var(axis=1, keepdims=True) + 1e-8

            out = np.zeros_like(x)
            for t in range(x.shape[1]):
                mean = (1 - factor_new) * mean + factor_new * x[:, t:t+1]
                var  = (1 - factor_new) * var  + factor_new * (x[:, t:t+1] - mean)**2
                out[:, t] = (x[:, t] - mean.squeeze()) / (np.sqrt(var.squeeze()) + 1e-8)
            X_out[trial] = out
        return X_out.astype(np.float32)

    @staticmethod
    def channel_zscore(X_train: np.ndarray,
                        X_test: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """Fit on train, apply to both. No leakage."""
        mean = X_train.mean(axis=(0, 2), keepdims=True)
        std  = X_train.std(axis=(0, 2), keepdims=True) + 1e-8
        return (X_train - mean) / std, (X_test - mean) / std

    def process(self, X_train, y_train, X_test, y_test,
                use_ems=True, reject=True):
        # 1. Artifact rejection (train only)
        if reject:
            X_train, y_train = self.reject_bad_trials(X_train, y_train)

        # 2. Common Average Reference
        X_train = self.common_average_reference(X_train)
        X_test  = self.common_average_reference(X_test)

        # 3. Exponential Moving Standardization
        if use_ems:
            print("    Applying EMS to train...", end=" ")
            X_train = self.exponential_moving_standardize(X_train)
            print("done. Test...", end=" ")
            X_test  = self.exponential_moving_standardize(X_test)
            print("done.")
        else:
            # Fallback: global channel z-score
            X_train, X_test = self.channel_zscore(X_train, X_test)

        return (X_train.astype(np.float32), y_train,
                X_test.astype(np.float32),  y_test)

print("✅ EEGPreprocessor defined.")

✅ EEGPreprocessor defined.


In [15]:
# ============================================================
# CELL 7 (FIXED): EEG Dataset with Corrected Augmentation
# Bug fixed: channel dropout was indexing wrong dimension
# x shape inside __getitem__: (1, n_channels, n_times)
# Correct channel zeroing: x[:, ch_idx, :] = 0.0  (dim 1)
# ============================================================

class EEGDataset(Dataset):
    """
    Input  X : (trials, channels, time)
    Output x : (1, channels, time) -- EEGNet expects (batch, 1, ch, time)
    """
    def __init__(self, X: np.ndarray, y: np.ndarray, augment: bool = False):
        # Add channel dimension: (trials, 1, channels, time)
        self.X      = torch.FloatTensor(X[:, np.newaxis, :, :])
        self.y      = torch.LongTensor(y)
        self.augment = augment
        self.n_ch   = X.shape[1]   # number of EEG channels (22)
        self.n_time = X.shape[2]   # number of time samples

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx].clone()   # shape: (1, n_channels, n_time)
        if self.augment:
            x = self._augment(x)
        return x, self.y[idx]

    def _augment(self, x: torch.Tensor) -> torch.Tensor:
        # x shape: (1, n_channels, n_time)
        # dim 0 = size 1  (singleton)
        # dim 1 = size 22 (channels)
        # dim 2 = size 875 (time)

        # 1. Gaussian noise
        if random.random() < 0.5:
            x = x + torch.randn_like(x) * 0.03

        # 2. Random amplitude scaling
        if random.random() < 0.5:
            scale = random.uniform(0.85, 1.15)
            x = x * scale

        # 3. Random temporal shift (+/- 40ms = +/- 10 samples)
        if random.random() < 0.4:
            shift = random.randint(-10, 10)
            x = torch.roll(x, shift, dims=2)   # dim 2 = time

        # 4. Channel dropout (FIXED -- index dim 1 = channels)
        if random.random() < 0.3:
            n_drop = random.randint(1, 2)
            ch_idx = random.sample(range(self.n_ch), n_drop)
            x[:, ch_idx, :] = 0.0   # dim 1 = channels

        # 5. Temporal segment masking
        if random.random() < 0.2:
            max_start = max(1, self.n_time - 50)
            start     = random.randint(0, max_start)
            x[:, :, start:start + 50] = 0.0   # dim 2 = time

        return x


def get_weighted_sampler(y: np.ndarray) -> WeightedRandomSampler:
    counts  = np.bincount(y)
    weights = 1.0 / counts[y]
    return WeightedRandomSampler(
        torch.FloatTensor(weights), len(y), replacement=True
    )


# Quick sanity check
print("Running augmentation sanity check...")
_X_dummy = np.random.randn(10, cfg.N_CHANNELS, cfg.N_SAMPLES).astype(np.float32)
_y_dummy = np.array([0, 1, 2, 3, 0, 1, 2, 3, 0, 1])
_ds      = EEGDataset(_X_dummy, _y_dummy, augment=True)

for i in range(len(_ds)):
    _x, _lbl = _ds[i]
    assert _x.shape == (1, cfg.N_CHANNELS, cfg.N_SAMPLES), \
        "Wrong shape: {}".format(_x.shape)

print("EEGDataset verified.")
print("  Output shape : (1, {}, {})".format(cfg.N_CHANNELS, cfg.N_SAMPLES))
print("  Augmentations: Noise | Scaling | Shift | ChannelDrop | SegmentMask")
del _X_dummy, _y_dummy, _ds

Running augmentation sanity check...
EEGDataset verified.
  Output shape : (1, 22, 875)
  Augmentations: Noise | Scaling | Shift | ChannelDrop | SegmentMask


In [16]:
# ============================================================
# CELL 8: Enhanced EEGNet Architecture
# ============================================================
class ConstrainedConv2d(nn.Conv2d):
    """Depthwise conv with max-norm constraint on weights (EEGNet paper)."""
    def __init__(self, *args, max_norm: float = 1.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.max_norm = max_norm

    def forward(self, x):
        self.weight.data = torch.renorm(
            self.weight.data, p=2, dim=0, maxnorm=self.max_norm
        )
        return super().forward(x)


class ConstrainedLinear(nn.Linear):
    """Dense layer with max-norm constraint."""
    def __init__(self, *args, max_norm: float = 0.25, **kwargs):
        super().__init__(*args, **kwargs)
        self.max_norm = max_norm

    def forward(self, x):
        self.weight.data = torch.renorm(
            self.weight.data, p=2, dim=0, maxnorm=self.max_norm
        )
        return super().forward(x)


class EEGNet(nn.Module):
    """
    Enhanced EEGNet with:
    - Max-norm constraints (original paper requirement)
    - Larger F1/F2 filters
    - Skip-connection variant in separable block
    - Additional BN after separable conv
    """
    def __init__(self, cfg: Config):
        super().__init__()
        C  = cfg.N_CHANNELS
        T  = cfg.N_SAMPLES
        F1 = cfg.F1
        D  = cfg.D
        F2 = cfg.F2
        K  = cfg.KERNEL_SIZE
        p  = cfg.DROPOUT

        # ── Block 1: Temporal + Depthwise Spatial ─────────
        self.block1 = nn.Sequential(
            # Temporal convolution
            nn.Conv2d(1, F1, kernel_size=(1, K), padding=(0, K//2), bias=False),
            nn.BatchNorm2d(F1),
            # Depthwise spatial filter (max-norm constrained)
            ConstrainedConv2d(F1, F1*D, kernel_size=(C, 1),
                              groups=F1, bias=False, max_norm=1.0),
            nn.BatchNorm2d(F1*D),
            nn.ELU(),
            nn.AvgPool2d((1, cfg.POOL1)),
            nn.Dropout2d(p),          # Dropout2d drops entire feature maps
        )

        # ── Block 2: Separable Convolution ────────────────
        self.block2 = nn.Sequential(
            # Depthwise
            nn.Conv2d(F1*D, F1*D, kernel_size=(1, 16),
                      padding=(0, 8), groups=F1*D, bias=False),
            # Pointwise
            nn.Conv2d(F1*D, F2, kernel_size=(1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1, cfg.POOL2)),
            nn.Dropout(p),
        )

        # ── Block 3: Additional Separable (performance boost) ─
        self.block3 = nn.Sequential(
            nn.Conv2d(F2, F2, kernel_size=(1, 8),
                      padding=(0, 4), groups=F2, bias=False),
            nn.Conv2d(F2, F2, kernel_size=(1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1, 2)),
            nn.Dropout(p),
        )

        # Compute flat size dynamically
        self._flat = self._flat_size(C, T)

        # ── Classifier (max-norm constrained) ─────────────
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self._flat, 128),
            nn.ELU(),
            nn.Dropout(p * 0.5),
            ConstrainedLinear(128, cfg.N_CLASSES, max_norm=0.25),
        )

        self._init_weights()

    def _flat_size(self, C, T):
        x = torch.zeros(1, 1, C, T)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        return x.numel()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.classifier(x)
        return x

    def get_features(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        return x.view(x.size(0), -1)


# Quick check
_model = EEGNet(cfg).to(cfg.DEVICE)
_dummy = torch.zeros(2, 1, cfg.N_CHANNELS, cfg.N_SAMPLES).to(cfg.DEVICE)
_out   = _model(_dummy)
total  = sum(p.numel() for p in _model.parameters())
print(f"✅ EEGNet built successfully.")
print(f"   Input  : {list(_dummy.shape)}")
print(f"   Output : {list(_out.shape)}")
print(f"   Params : {total:,}")
del _model, _dummy, _out

✅ EEGNet built successfully.
   Input  : [2, 1, 22, 875]
   Output : [2, 4]
   Params : 62,756


In [17]:
# ============================================================
# CELL 9: Training Engine (Mixup + Cosine Annealing + Early Stop)
# ============================================================
class EarlyStopping:
    def __init__(self, patience=60, min_delta=1e-4):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best       = -np.inf
        self.counter    = 0
        self.best_state = None

    def step(self, val_acc, model):
        if val_acc > self.best + self.min_delta:
            self.best       = val_acc
            self.counter    = 0
            self.best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            return False
        self.counter += 1
        return self.counter >= self.patience

    def restore(self, model):
        if self.best_state:
            model.load_state_dict(self.best_state)


def mixup_data(x, y, alpha=0.2):
    """Mixup augmentation — interpolates pairs of samples."""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    idx    = torch.randperm(x.size(0), device=x.device)
    x_mix  = lam * x + (1 - lam) * x[idx]
    return x_mix, y, y[idx], lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


class Trainer:
    def __init__(self, model, cfg: Config):
        self.model     = model.to(cfg.DEVICE)
        self.cfg       = cfg
        self.device    = cfg.DEVICE
        self.history   = dict(train_loss=[], val_loss=[], train_acc=[], val_acc=[])

        self.criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
        self.optimizer = optim.AdamW(
            model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY
        )
        # Cosine annealing with warm restarts
        self.scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            self.optimizer, T_0=50, T_mult=2, eta_min=cfg.MIN_LR
        )
        self.es = EarlyStopping(patience=cfg.PATIENCE)

    def _epoch(self, loader, train=True, use_mixup=False):
        self.model.train(train)
        total_loss, correct, total = 0., 0, 0
        with torch.set_grad_enabled(train):
            for X_b, y_b in loader:
                X_b, y_b = X_b.to(self.device), y_b.to(self.device)
                if train and use_mixup and random.random() < 0.5:
                    X_b, y_a, y_b_mix, lam = mixup_data(X_b, y_b, alpha=0.2)
                    logits = self.model(X_b)
                    loss   = mixup_criterion(self.criterion, logits, y_a, y_b_mix, lam)
                else:
                    logits = self.model(X_b)
                    loss   = self.criterion(logits, y_b)

                if train:
                    self.optimizer.zero_grad()
                    loss.backward()
                    nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                    self.optimizer.step()

                total_loss += loss.item() * len(y_b)
                correct    += (logits.argmax(1) == y_b).sum().item()
                total      += len(y_b)

        return total_loss / total, correct / total

    def fit(self, train_loader, val_loader, verbose=True):
        for epoch in range(1, self.cfg.EPOCHS + 1):
            tr_l, tr_a = self._epoch(train_loader, train=True,  use_mixup=True)
            vl_l, vl_a = self._epoch(val_loader,   train=False, use_mixup=False)

            self.scheduler.step(epoch)
            self.history["train_loss"].append(tr_l)
            self.history["val_loss"].append(vl_l)
            self.history["train_acc"].append(tr_a)
            self.history["val_acc"].append(vl_a)

            if verbose and (epoch % 50 == 0 or epoch == 1):
                lr = self.optimizer.param_groups[0]["lr"]
                print(f"  Ep {epoch:04d} | "
                      f"tr_loss={tr_l:.4f} tr_acc={tr_a:.4f} | "
                      f"vl_loss={vl_l:.4f} vl_acc={vl_a:.4f} | lr={lr:.2e}")

            if self.es.step(vl_a, self.model):
                if verbose:
                    print(f"  ⏹ Early stop at epoch {epoch}. Best val acc: {self.es.best:.4f}")
                break

        self.es.restore(self.model)
        return self.history

    @torch.no_grad()
    def predict_proba(self, loader) -> Tuple[np.ndarray, np.ndarray]:
        self.model.eval()
        probs_all, labels_all = [], []
        for X_b, y_b in loader:
            logits = self.model(X_b.to(self.device))
            probs  = F.softmax(logits, dim=1).cpu().numpy()
            probs_all.extend(probs)
            labels_all.extend(y_b.numpy())
        return np.array(probs_all), np.array(labels_all)


print("✅ Trainer with Mixup + Cosine Annealing defined.")

✅ Trainer with Mixup + Cosine Annealing defined.


In [18]:
# ============================================================
# CELL 10: Ensemble Training (N models per subject)
# ============================================================
def train_ensemble_subject(
    X_train, y_train,
    X_test,  y_test,
    subject_id: int,
    cfg: Config,
    n_models: int = 5,
) -> Dict:
    """
    Train N independent EEGNet models with different seeds.
    Aggregate predictions via soft voting (average probabilities).
    This is the #1 technique for pushing accuracy above 80%.
    """
    print(f"\n{'='*58}")
    print(f"  Subject {subject_id:02d} — Ensemble of {n_models} models")
    print(f"{'='*58}")

    preproc = EEGPreprocessor()
    X_tr, y_tr, X_te, y_te = preproc.process(
        X_train.copy(), y_train.copy(),
        X_test.copy(),  y_test.copy(),
        use_ems=True, reject=True
    )

    # ── DataLoaders ───────────────────────────────────────
    test_ds     = EEGDataset(X_te, y_te, augment=False)
    test_loader = DataLoader(test_ds, batch_size=cfg.BATCH_SIZE,
                             shuffle=False, num_workers=2, pin_memory=True)

    all_probs  = []
    best_accs  = []

    for m_idx in range(n_models):
        seed = cfg.SEED + m_idx * 13
        seed_everything(seed)
        print(f"\n  ── Model {m_idx+1}/{n_models} (seed={seed}) ──────────────")

        # ── Train/Val split for this model ────────────────
        val_frac = 0.15
        n_val    = max(1, int(len(X_tr) * val_frac))
        indices  = np.random.permutation(len(X_tr))
        tr_idx, vl_idx = indices[n_val:], indices[:n_val]

        tr_ds = EEGDataset(X_tr[tr_idx], y_tr[tr_idx], augment=True)
        vl_ds = EEGDataset(X_tr[vl_idx], y_tr[vl_idx], augment=False)

        sampler    = get_weighted_sampler(y_tr[tr_idx])
        tr_loader  = DataLoader(tr_ds, batch_size=cfg.BATCH_SIZE,
                                sampler=sampler, num_workers=2, pin_memory=True)
        vl_loader  = DataLoader(vl_ds, batch_size=cfg.BATCH_SIZE,
                                shuffle=False, num_workers=2, pin_memory=True)

        model   = EEGNet(cfg).to(cfg.DEVICE)
        trainer = Trainer(model, cfg)
        trainer.fit(tr_loader, vl_loader, verbose=True)

        # ── Test predictions ──────────────────────────────
        probs, labels = trainer.predict_proba(test_loader)
        acc   = accuracy_score(labels, probs.argmax(1))
        best_accs.append(acc)
        all_probs.append(probs)

        # ── Save model ────────────────────────────────────
        torch.save(
            model.state_dict(),
            cfg.MODEL_DIR / f"eegnet_s{subject_id:02d}_m{m_idx+1}.pt"
        )
        print(f"  Model {m_idx+1} individual test acc: {acc:.4f}")

    # ── Soft Vote Ensemble ────────────────────────────────
    ensemble_probs = np.mean(all_probs, axis=0)      # Average probabilities
    y_pred         = ensemble_probs.argmax(1)
    y_true         = labels

    acc   = accuracy_score(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    cm    = confusion_matrix(y_true, y_pred)
    rep   = classification_report(y_true, y_pred,
                                   target_names=cfg.CLASS_NAMES,
                                   output_dict=True)

    print(f"\n  ✅ Ensemble Test Accuracy : {acc:.4f} ({acc*100:.2f}%)")
    print(f"  ✅ Ensemble Kappa        : {kappa:.4f}")
    print(f"  Individual accs: {[f'{a:.3f}' for a in best_accs]}")

    return dict(
        accuracy=acc, kappa=kappa, cm=cm, report=rep,
        y_true=y_true, y_pred=y_pred,
        ensemble_probs=ensemble_probs,
        individual_accs=best_accs
    )

print("✅ Ensemble training function defined.")

✅ Ensemble training function defined.


In [19]:
# ============================================================
# CELL 11: Visualization Utilities
# ============================================================
class Visualizer:
    def __init__(self, results_dir: Path):
        self.rd = results_dir

    def confusion_matrix(self, cm, subject_id, acc, kappa):
        fig, ax = plt.subplots(figsize=(7, 6))
        pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
        sns.heatmap(pct, annot=True, fmt=".1f", cmap="Blues",
                    xticklabels=cfg.CLASS_NAMES, yticklabels=cfg.CLASS_NAMES,
                    ax=ax, linewidths=0.5, cbar_kws={"label": "%"})
        ax.set_title(f"Confusion Matrix — Subject {subject_id:02d} "
                     f"(Acc={acc*100:.1f}%, κ={kappa:.3f})",
                     fontsize=13, fontweight="bold")
        ax.set_ylabel("True"); ax.set_xlabel("Predicted")
        plt.tight_layout()
        plt.savefig(self.rd / f"cm_s{subject_id:02d}.png", dpi=150)
        plt.show()

    def per_class_bars(self, report, subject_id):
        metrics = ["precision", "recall", "f1-score"]
        vals    = {m: [report[c][m] for c in cfg.CLASS_NAMES] for m in metrics}
        x, w    = np.arange(4), 0.25
        colors  = ["steelblue", "darkorange", "seagreen"]
        fig, ax = plt.subplots(figsize=(10, 5))
        for i, (m, c) in enumerate(zip(metrics, colors)):
            ax.bar(x + i*w, vals[m], w, label=m.capitalize(),
                   color=c, alpha=0.85, edgecolor="black")
        ax.set_xticks(x + w); ax.set_xticklabels(cfg.CLASS_NAMES)
        ax.set_ylim(0, 1.15); ax.set_ylabel("Score")
        ax.set_title(f"Per-Class Metrics — Subject {subject_id:02d}",
                     fontsize=13, fontweight="bold")
        ax.legend(); ax.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        plt.savefig(self.rd / f"class_s{subject_id:02d}.png", dpi=150)
        plt.show()

    def subject_summary(self, results: Dict):
        sids   = sorted(results.keys())
        accs   = [results[s]["accuracy"]*100 for s in sids]
        kappas = [results[s]["kappa"]        for s in sids]
        labels = [f"S{s:02d}" for s in sids]

        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        for ax, vals, ylabel, color, thresh in zip(
            axes,
            [accs, kappas],
            ["Accuracy (%)", "Cohen's Kappa"],
            ["steelblue", "darkorange"],
            [80.0, 0.73]   # 80% and equivalent kappa targets
        ):
            bars = ax.bar(labels, vals, color=color, alpha=0.8, edgecolor="black")
            ax.axhline(np.mean(vals), color="red", linestyle="--", lw=2,
                       label=f"Mean = {np.mean(vals):.2f}")
            ax.axhline(thresh, color="green", linestyle=":", lw=2,
                       label=f"Target = {thresh}")
            ax.set_title(f"Per-Subject {ylabel}", fontsize=13, fontweight="bold")
            ax.set_ylabel(ylabel); ax.legend(); ax.grid(axis="y", alpha=0.3)
            for bar, v in zip(bars, vals):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                        f"{v:.1f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

        plt.suptitle("EEGNet Ensemble — BCI Competition IV 2a Results",
                     fontsize=15, fontweight="bold")
        plt.tight_layout()
        plt.savefig(self.rd / "subject_summary.png", dpi=150)
        plt.show()

viz = Visualizer(cfg.RESULTS_DIR)
print("✅ Visualizer ready.")

✅ Visualizer ready.


In [ ]:
# ============================================================
# CELL 12: Run Full Experiment — All 9 Subjects
# ============================================================
# ⏱ Expected time: ~20-35 minutes on GPU (T4/V100)
# 🎯 Target: >80% mean accuracy across 9 subjects

all_results = {}

for subject_id in cfg.SUBJECTS:
    data = all_data.get(subject_id)
    if data is None:
        print(f"⚠️ Subject {subject_id:02d} skipped (data not loaded).")
        continue

    results = train_ensemble_subject(
        X_train=data["X_train"], y_train=data["y_train"],
        X_test=data["X_test"],   y_test=data["y_test"],
        subject_id=subject_id,
        cfg=cfg,
        n_models=cfg.N_ENSEMBLE,
    )
    all_results[subject_id] = results

    # ── Per-subject visualizations ────────────────────────
    viz.confusion_matrix(
        results["cm"], subject_id,
        results["accuracy"], results["kappa"]
    )
    viz.per_class_bars(results["report"], subject_id)

print("\n✅ All subjects completed.")


  Subject 01 — Ensemble of 5 models
    Applying EMS to train... done. Test... done.

  ── Model 1/5 (seed=42) ──────────────
  Ep 0001 | tr_loss=1.3886 tr_acc=0.2490 | vl_loss=1.3878 vl_acc=0.2558 | lr=5.00e-04
  Ep 0050 | tr_loss=1.1480 tr_acc=0.4163 | vl_loss=1.0614 vl_acc=0.6512 | lr=5.00e-04
  Ep 0100 | tr_loss=0.9573 tr_acc=0.6449 | vl_loss=0.9078 vl_acc=0.6512 | lr=2.51e-04
  Ep 0150 | tr_loss=0.8216 tr_acc=0.5755 | vl_loss=0.8910 vl_acc=0.6977 | lr=5.00e-04
  Ep 0200 | tr_loss=0.7945 tr_acc=0.6694 | vl_loss=0.8572 vl_acc=0.7209 | lr=4.27e-04
  Ep 0250 | tr_loss=0.8686 tr_acc=0.5918 | vl_loss=0.8381 vl_acc=0.7209 | lr=2.51e-04
  ⏹ Early stop at epoch 256. Best val acc: 0.8372
  Model 1 individual test acc: 0.6215

  ── Model 2/5 (seed=55) ──────────────
  Ep 0001 | tr_loss=1.3942 tr_acc=0.2612 | vl_loss=1.3830 vl_acc=0.2791 | lr=5.00e-04
  Ep 0050 | tr_loss=1.1404 tr_acc=0.5347 | vl_loss=1.0774 vl_acc=0.4884 | lr=5.00e-04
  Ep 0100 | tr_loss=0.9725 tr_acc=0.5959 | vl_loss=0.988